# StoryBored を Colab / Kaggle で動かす(実験的)

GPU のないマシンからでも、このノートブック 1 つで **ComfyUI + StoryBored** をクラウド GPU 上に立ち上げて、ブラウザから普段どおり使えます。**実験的なサポート**です — プラットフォーム側の仕様変更で突然動かなくなることがあります。

## 必要なもの

- **ngrok アカウントと authtoken**(無料枠で OK)— [authtoken の確認はこちら](https://dashboard.ngrok.com/get-started/your-authtoken)。ngrok の無料枠は **固定ドメイン** を 1 個取得できます。持っている場合はシークレット `NGROK_DOMAIN` も設定すると、セッションをまたいで同じ URL になります。
- **GPU セッション** — 静止画生成なら T4 16 GB で足ります。どのエンジンがどの GPU で動くかは、README の「Colab / Kaggle で動かす(実験的)」のハードウェア適合表を参照してください。

## Colab の場合

- [このボタンで開く](https://colab.research.google.com/github/ma-team333/storybored/blob/ja/notebooks/storybored-cloud.ipynb)か、GitHub からノートブックをアップロードします。
- ランタイムは**無料の T4** で足ります(ランタイム → ランタイムのタイプを変更 → T4 GPU)。
- authtoken は左ペインの**シークレット**(`google.colab.userdata`)に `NGROK_AUTHTOKEN` という名前で保存してください。未設定のときは実行時に入力を求めます。

## Kaggle の場合

- **File → Import Notebook** でこの .ipynb を取り込んでください。
- **Settings → Internet を ON** にしないと clone も pip も通りません。
- **Settings → Accelerator で GPU T4 x2** を選択します。
- authtoken は **Add-ons → Secrets**(`UserSecretsClient`)に `NGROK_AUTHTOKEN` という名前で保存してください。
- セッションは最長 **12 時間**、GPU 利用は**週 30 時間の割り当て**制限があります。

## データは消えます(重要)

ノートブックのディスクは**エフェメラル**です。セッションが切れると `data/`(ボード・スチル・クリップ・アニマティック)はすべて消失します。こまめに最後の**データ退避セル**を実行するか、UI の**書き出し**(`.storybored` ファイル)でプロジェクト単位の保存をしてください。

## 流れ

1. 下のセルを**上から順に**実行します(環境検出 → clone とビルド → ComfyUI → StoryBored → ngrok トンネル)。
2. 最後のセルに表示される**トークン付き URL** をブラウザで開きます(以後はドメイン + トークン入力)。
3. 初回は Settings の **Download** からモデルをダウンロードします。ノートブックは `COMFY_MODELS_DIR` を ComfyUI の models フォルダに向けて起動するので、アプリ内モデルダウンローダーがそのまま使えます(Z-Image Turbo 一式で約 12 GB)。

In [ ]:
# 実行環境の検出と、ngrok authtoken の取得
import os
from getpass import getpass
from pathlib import Path

IN_COLAB = Path('/content').exists() and not Path('/kaggle').exists()
IN_KAGGLE = Path('/kaggle').exists()
ENV_NAME = 'Colab' if IN_COLAB else ('Kaggle' if IN_KAGGLE else '不明な環境(ローカル?)')
print(f'検出された環境: {ENV_NAME}')

# ノートブックの作業ディレクトリ(セッション終了で消えます)
WORK = Path('/content') if IN_COLAB else Path('/kaggle/working')
print(f'作業ディレクトリ: {WORK}')

def get_secret(name, required=True):
    # ノートブックのシークレット機能から取得し、だめなら getpass で手動入力。
    # required=False は任意の値(固定ドメインなど)用で、無ければ空文字。
    try:
        if IN_COLAB:
            from google.colab import userdata
            return userdata.get(name)
        if IN_KAGGLE:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(name)
    except Exception as exc:
        if not required:
            return ''
        print(f'シークレット {name} が取得できなかったため手動入力します ({type(exc).__name__})')
    if not required:
        return ''
    return getpass(f'{name} を入力してください: ')

def port_open(port):
    import socket
    with socket.socket() as sock:
        sock.settimeout(0.5)
        return sock.connect_ex(('127.0.0.1', port)) == 0

NGROK_AUTHTOKEN = (os.environ.get('NGROK_AUTHTOKEN') or get_secret('NGROK_AUTHTOKEN')).strip()
assert NGROK_AUTHTOKEN, 'ngrok authtoken が必要です (https://dashboard.ngrok.com/get-started/your-authtoken)'
os.environ['NGROK_AUTHTOKEN'] = NGROK_AUTHTOKEN
print('NGROK_AUTHTOKEN を設定しました ✓')

In [ ]:
# StoryBored の clone、Python 依存の導入、フロントエンドのビルド(全体で数分かかります)
import glob
import subprocess

REPO = WORK / 'storybored'

def run(cmd, cwd=None):
    print('$', cmd)
    result = subprocess.run(cmd, shell=True, cwd=cwd)
    if result.returncode != 0:
        raise RuntimeError(f'コマンドが失敗しました: {cmd}')

if not (REPO / '.git').exists():
    run(f'git clone -b ja https://github.com/ma-team333/storybored {REPO}')

# バックエンド(依存は backend/pyproject.toml)+ トンネル用の pyngrok
run('pip install -q -e backend pyngrok', cwd=REPO)

def node_major():
    try:
        version = subprocess.check_output(['node', '-v'], text=True).strip().lstrip('v')
        return int(version.split('.')[0])
    except Exception:
        return 0

# Node 20+(engines 制約)を確認し、無ければ nvm で導入する
if node_major() < 20:
    run('curl -fsSL https://raw.githubusercontent.com/nvm-sh/nvm/v0.40.3/install.sh | bash')
    nvm_dir = Path.home() / '.nvm'
    run(f'. {nvm_dir}/nvm.sh && nvm install 20')
    node_bins = sorted(glob.glob(f'{nvm_dir}/versions/node/v20*/bin'))
    assert node_bins, 'Node 20 の導入に失敗しました'
    os.environ['PATH'] = node_bins[-1] + os.pathsep + os.environ['PATH']
print('node', subprocess.check_output(['node', '-v'], text=True).strip())

# フロントエンドをビルドする(バックエンドが frontend/dist を静的配信する)
run('npm ci --no-fund --no-audit && npm run build', cwd=REPO / 'frontend')
print('フロントエンドのビルド完了 ✓')

In [ ]:
# ComfyUI の clone・依存導入・起動。/system_stats が応答するまで待つ
import sys
import time
import urllib.request

COMFY = WORK / 'ComfyUI'

if not (COMFY / 'main.py').exists():
    run(f'git clone --depth 1 https://github.com/comfyanonymous/ComfyUI {COMFY}')
    run('pip install -q -r requirements.txt', cwd=COMFY)

def wait_for(url, headers=None, timeout_s=600, interval_s=5):
    # url が 200 を返すまで待つ(タイムアウトで False)
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        try:
            request = urllib.request.Request(url, headers=headers or {})
            with urllib.request.urlopen(request, timeout=10) as response:
                if response.status == 200:
                    return True
        except Exception:
            pass
        time.sleep(interval_s)
    return False

def show_log(path, lines=40):
    import subprocess
    completed = subprocess.run(['tail', '-n', str(lines), str(path)],
                               capture_output=True, text=True)
    print(completed.stdout)

import subprocess as _sp

if port_open(8188):
    print('ComfyUI は既に 8188 で応答しています。起動はスキップします。')
else:
    comfy_log = open(COMFY / 'comfyui.log', 'ab')
    _sp.Popen(
        [sys.executable, 'main.py', '--listen', '127.0.0.1', '--port', '8188',
         '--disable-auto-launch'],
        cwd=COMFY, stdout=comfy_log, stderr=_sp.STDOUT, start_new_session=True)
    print('ComfyUI を起動中 … (初回は初期化に数分かかることがあります)')
    if not wait_for('http://127.0.0.1:8188/system_stats', timeout_s=600):
        show_log(COMFY / 'comfyui.log')
        raise RuntimeError('ComfyUI が起動しませんでした。上のログを確認してください。')
    print('ComfyUI 起動 ✓ (http://127.0.0.1:8188)')

In [ ]:
# StoryBored の起動。STORYBORED_TOKEN は secrets モジュールで自動生成して表示する
# (セルを再実行するとトークンは新しい値に変わります — 表示された URL を使い直してください)
import secrets

def token_in_env():
    env_path = REPO / '.env'
    if not env_path.exists():
        return ''
    for line in env_path.read_text(encoding='utf-8').splitlines():
        if line.startswith('STORYBORED_TOKEN='):
            return line.split('=', 1)[1].strip()
    return ''

if port_open(8600) and token_in_env():
    STORYBORED_TOKEN = token_in_env()
    print('StoryBored は既に 8600 で応答しています。起動はスキップします。')
    print(f'StoryBored アクセストークン: {STORYBORED_TOKEN}')
else:
    STORYBORED_TOKEN = secrets.token_urlsafe(24)
    print(f'StoryBored アクセストークン: {STORYBORED_TOKEN}')

    # .env は PROJECT_ROOT(repo ルート)に置く。COMFY_MODELS_DIR を ComfyUI の models に
    # 向けると、アプリ内モデルダウンローダー(Settings の Download)とサイズ警告が有効になる
    (REPO / '.env').write_text(
        'STORYBORED_HOST=127.0.0.1\n'
        'STORYBORED_PORT=8600\n'
        'DATA_DIR=./data\n'
        'COMFYUI_URL=http://127.0.0.1:8188\n'
        f'COMFY_MODELS_DIR={COMFY}/models\n'
        f'STORYBORED_TOKEN={STORYBORED_TOKEN}\n',
        encoding='utf-8')

    server_env = dict(os.environ,
                      STORYBORED_TOKEN=STORYBORED_TOKEN,
                      COMFYUI_URL='http://127.0.0.1:8188',
                      COMFY_MODELS_DIR=f'{COMFY}/models')
    storybored_log = open(REPO / 'storybored.log', 'ab')
    subprocess.Popen(
        [sys.executable, '-m', 'storybored'],
        cwd=REPO, env=server_env, stdout=storybored_log,
        stderr=subprocess.STDOUT, start_new_session=True)

    print('StoryBored を起動中 …')
    if not wait_for('http://127.0.0.1:8600/api/health',
                    headers={'Authorization': f'Bearer {STORYBORED_TOKEN}'},
                    timeout_s=180):
        show_log(REPO / 'storybored.log')
        raise RuntimeError('StoryBored が起動しませんでした。上のログを確認してください。')
    print('StoryBored 起動 ✓ (http://127.0.0.1:8600)')

In [ ]:
# ngrok で 8600 番ポートを公開する
# 注意: cloudflared のクイックトンネル(trycloudflare.com)は SSE(/api/events)に
# 非対応のため使えません — StoryBored は ngrok(pyngrok)を使います
from pyngrok import ngrok

ngrok.set_auth_token(os.environ['NGROK_AUTHTOKEN'])

# ngrok 無料枠の固定ドメインを持っていれば、シークレット NGROK_DOMAIN に入れると
# セッションをまたいで同じ URL になります
domain = (os.environ.get('NGROK_DOMAIN') or get_secret('NGROK_DOMAIN', required=False) or '').strip() or None
if domain:
    tunnel = ngrok.connect(8600, 'http', domain=domain)
else:
    tunnel = ngrok.connect(8600, 'http')
PUBLIC_URL = tunnel.public_url.rstrip('/')

print('=== ブラウザで開く URL ===')
print(f'トークン付き URL(まずはこちら)   : {PUBLIC_URL}/?token={STORYBORED_TOKEN}')
print(f'公開 URL(以後はこちら+トークン) : {PUBLIC_URL}')
print()
print('ブックマークや共有は『ドメイン + トークン』で。トークンは StoryBored 起動セルが表示した値です。')

## データ退避(セッションを閉じる前に必ず!)

ノートブックのディスクはセッションと一緒に消えます。このセルで `data/` フォルダ一式を zip に退避します — **Colab は Google Drive へ**、**Kaggle はブラウザへのダウンロード**で保存します。

- プロジェクト単位なら、UI の**書き出し**(`.storybored` ファイル)でも保存できます。読み込みは別マシンの StoryBored でもそのまま使えます。
- 次のセッションでは、最後の**復元セル**に zip のパスを指定して戻してください。

In [ ]:
# DATA_DIR(既定 ./data)を 1 つの zip に退避する
import shutil

data_dir = REPO / 'data'
assert data_dir.exists(), 'data/ フォルダがありません(まだ何も生成していない?)'
backup_zip = shutil.make_archive(str(WORK / 'storybored-data-backup'), 'zip',
                                 root_dir=REPO, base_dir='data')
size_mb = Path(backup_zip).stat().st_size / 1_000_000
print(f'退避を作成しました: {backup_zip} ({size_mb:.1f} MB)')

if IN_COLAB:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
    dest_dir = Path('/content/drive/MyDrive/storybored-backups')
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest = dest_dir / Path(backup_zip).name
    shutil.copy2(backup_zip, dest)
    print(f'Google Drive に保存しました: {dest}')
elif IN_KAGGLE:
    from IPython.display import FileLink, display
    display(FileLink(Path(backup_zip).name))
    print('リンクをクリックしてブラウザに保存してください(右ペイン Output にも残ります)。')
else:
    print(f'{backup_zip} を手元に保存してください。')

In [ ]:
# 次のセッションで復元するときだけ実行する。BACKUP_ZIP に退避 zip のパスを入れる
#   Colab : /content/drive/MyDrive/storybored-backups/storybored-data-backup.zip
#   Kaggle: 退避した zip を Input にアップロードして
#           /kaggle/input/<データセット名>/storybored-data-backup.zip を指定
import zipfile

BACKUP_ZIP = ''

assert BACKUP_ZIP, 'BACKUP_ZIP に退避 zip のパスを設定してください'
if (IN_COLAB and BACKUP_ZIP.startswith('/content/drive')
        and not Path('/content/drive/MyDrive').exists()):
    from google.colab import drive
    drive.mount('/content/drive')

REPO = WORK / 'storybored'
if not (REPO / '.git').exists():
    raise RuntimeError('StoryBored がありません。上のセル(環境検出 → clone/ビルド)を先に実行してください')
with zipfile.ZipFile(BACKUP_ZIP) as archive:
    archive.extractall(REPO)
assert (REPO / 'data').exists(), 'zip の中に data/ フォルダが見つかりません'
print(f'復元しました: {REPO}/data')
print('続きは ComfyUI 起動セル → StoryBored 起動セル → ngrok セルを順に実行してください')